# 现代循环神经网络

介绍 GRU、LSTM、深度 RNN 和双向 RNN。

In [13]:
import torch
from torch import nn
from torch.nn import functional as F
import collections
import random
import re
import math

def read_time_machine():
    with open('time_machine.txt', 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

def tokenize(lines, token='word'):
    return [line.split() for line in lines] if token == 'word' else [list(line) for line in lines]

class Vocab:
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None: tokens = []
        if reserved_tokens is None: reserved_tokens = []
        counter = collections.Counter(t for line in tokens for t in line)
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.unk, uniq_tokens = 0, ['<unk>'] + reserved_tokens
        uniq_tokens += [t for t, f in self.token_freqs if f >= min_freq and t not in uniq_tokens]
        self.idx_to_token, self.token_to_idx = [], {}
        for token in uniq_tokens:
            self.idx_to_token.append(token)
            self.token_to_idx[token] = len(self.idx_to_token) - 1
    def __getitem__(self, items):
        if isinstance(items, str): return self.token_to_idx.get(items, self.unk)
        return [self.__getitem__(i) for i in items]
    def __len__(self): return len(self.idx_to_token)

def load_corpus_time_machine(max_tokens=-1):
    lines = read_time_machine()
    tokens = tokenize(lines, token='char')
    vocab = Vocab(tokens)
    corpus = [vocab[t] for line in tokens for t in line]
    if max_tokens > 0: corpus = corpus[:max_tokens]
    return corpus, vocab

def seq_data_iter_random(corpus, batch_size, num_steps):
    corpus = corpus[random.randint(0, num_steps - 1):]
    num_subseqs = (len(corpus) - 1) // num_steps
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)
    def data(pos): return corpus[pos:pos + num_steps]
    num_batches = num_subseqs // batch_size
    for i in range(0, batch_size * num_batches, batch_size):
        indices = initial_indices[i:i + batch_size]
        X = torch.tensor([data(j) for j in indices])
        Y = torch.tensor([data(j + 1) for j in indices])
        yield X, Y

class SeqDataLoader:
    def __init__(self, batch_size, num_steps, use_random=False, max_tokens=10000):
        self.data_iter_fn = seq_data_iter_random
        self.corpus, self.vocab = load_corpus_time_machine(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps
    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)

def load_data_time_machine(batch_size, num_steps, use_random=False, max_tokens=10000):
    data_iter = SeqDataLoader(batch_size, num_steps, use_random, max_tokens)
    return data_iter, data_iter.vocab

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

def predict(prefix, num_preds, net, vocab, device):
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])

def grad_clipping(net, theta):
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

def train_epoch(net, data_iter, loss, updater, device, use_random_iter):
    state = None
    metric = [0.0, 0.0]
    for X, Y in data_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(state, tuple):
                for s in state: s.detach_()
            else:
                state.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric[0] += l.item() * y.numel()
        metric[1] += y.numel()
    return math.exp(metric[0] / metric[1])

def train(net, data_iter, lr, num_epochs, device, vocab):
    loss = nn.CrossEntropyLoss()
    if isinstance(net, nn.Module):
        net.to(device)
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    for epoch in range(num_epochs):
        ppl = train_epoch(net, data_iter, loss, updater, device, False)
        if (epoch + 1) % 100 == 0:
            print(f'epoch {epoch + 1}, perplexity {ppl:.1f}')
    print(f'predict: {predict("time traveller ", 50, net, vocab, device)}')

device = torch.device('cuda')
print(f'Using device: {device}')

Using device: cuda


## 1. 门控循环单元（GRU）

GRU 通过重置门和更新门控制信息流，缓解 RNN 的梯度消失问题。

### 从零实现

In [8]:
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    W_xz = normal((num_inputs, num_hiddens))
    W_hz = normal((num_hiddens, num_hiddens))
    b_z = torch.zeros(num_hiddens, device=device)
    W_xr = normal((num_inputs, num_hiddens))
    W_hr = normal((num_hiddens, num_hiddens))
    b_r = torch.zeros(num_hiddens, device=device)
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def init_gru_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),)

def gru(inputs, state, params):
    W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        Z = torch.sigmoid((X @ W_xz) + (H @ W_hz) + b_z)
        R = torch.sigmoid((X @ W_xr) + (H @ W_hr) + b_r)
        H_tilde = torch.tanh((X @ W_xh) + ((R * H) @ W_hh) + b_h)
        H = Z * H + (1 - Z) * H_tilde
        Y = H @ W_hq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

class RNNModelScratch:
    def __init__(self, vocab_size, num_hiddens, device, get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn
    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)
    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)

batch_size, num_steps, num_hiddens = 32, 35, 256
data_iter, vocab = load_data_time_machine(batch_size, num_steps)
vocab_size = len(vocab)
net = RNNModelScratch(vocab_size, num_hiddens, device, get_params, init_gru_state, gru)
train(net, data_iter, 1, 500, device, vocab)

epoch 100, perplexity 9.2
epoch 200, perplexity 6.8
epoch 300, perplexity 4.6
epoch 400, perplexity 2.5
epoch 500, perplexity 1.8
predict: time traveller smiled round at us then still smiling faintly andw


### 简洁实现

使用 PyTorch 的 `nn.GRU`。

In [11]:
class RNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size).type(torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, batch_size, device):
        if not isinstance(self.rnn, nn.LSTM):
            return torch.zeros((self.num_directions * self.rnn.num_layers,
                               batch_size, self.num_hiddens), device=device)
        else:
            h = torch.zeros((self.num_directions * self.rnn.num_layers,
                             batch_size, self.num_hiddens), device=device)
            c = torch.zeros((self.num_directions * self.rnn.num_layers,
                             batch_size, self.num_hiddens), device=device)
            return (h, c)

gru_layer = nn.GRU(input_size=vocab_size, hidden_size=num_hiddens)
net = RNNModel(gru_layer, vocab_size)
train(net, data_iter, 1, 500, device, vocab)

epoch 100, perplexity 8.2
epoch 200, perplexity 5.1
epoch 300, perplexity 2.3
epoch 400, perplexity 1.8
epoch 500, perplexity 1.7
predict: time traveller smiled round at us then still smiling faintly andw


## 2. 长短期记忆网络（LSTM）

LSTM 引入记忆单元和三门控结构（输入门、遗忘门、输出门），更精细地管理长期记忆。

### 从零实现

LSTM 需要 14 组参数：输入门、遗忘门、输出门、候选记忆单元各 3 组，输出层 2 组。状态返回 `(H, C)` 元组。

In [14]:
def get_lstm_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    W_xi = normal((num_inputs, num_hiddens))
    W_hi = normal((num_hiddens, num_hiddens))
    b_i = torch.zeros(num_hiddens, device=device)
    W_xf = normal((num_inputs, num_hiddens))
    W_hf = normal((num_hiddens, num_hiddens))
    b_f = torch.zeros(num_hiddens, device=device)
    W_xo = normal((num_inputs, num_hiddens))
    W_ho = normal((num_hiddens, num_hiddens))
    b_o = torch.zeros(num_hiddens, device=device)
    W_xc = normal((num_inputs, num_hiddens))
    W_hc = normal((num_hiddens, num_hiddens))
    b_c = torch.zeros(num_hiddens, device=device)
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def init_lstm_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),
            torch.zeros((batch_size, num_hiddens), device=device))

def lstm(inputs, state, params):
    W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c, W_hq, b_q = params
    H, C = state
    outputs = []
    for X in inputs:
        I = torch.sigmoid((X @ W_xi) + (H @ W_hi) + b_i)
        F = torch.sigmoid((X @ W_xf) + (H @ W_hf) + b_f)
        O = torch.sigmoid((X @ W_xo) + (H @ W_ho) + b_o)
        C_tilde = torch.tanh((X @ W_xc) + (H @ W_hc) + b_c)
        C = F * C + I * C_tilde
        H = O * torch.tanh(C)
        Y = H @ W_hq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H, C)

net_lstm = RNNModelScratch(vocab_size, num_hiddens, device, get_lstm_params, init_lstm_state, lstm)
train(net_lstm, data_iter, 1, 500, device, vocab)

epoch 100, perplexity 10.7
epoch 200, perplexity 7.9
epoch 300, perplexity 5.7
epoch 400, perplexity 3.6
epoch 500, perplexity 2.3
predict: time traveller smiled are you so sure we converefte the peosery o


### 简洁实现

使用 PyTorch 的 `nn.LSTM`。

In [15]:
lstm_layer = nn.LSTM(input_size=vocab_size, hidden_size=num_hiddens)
net_lstm_con = RNNModel(lstm_layer, vocab_size)
train(net_lstm_con, data_iter, 1, 500, device, vocab)

epoch 100, perplexity 9.1
epoch 200, perplexity 5.9
epoch 300, perplexity 3.2
epoch 400, perplexity 2.1
epoch 500, perplexity 1.8
predict: time traveller but now you begin to overlook think and the little


## 3. 深度循环神经网络

堆叠多个 RNN 层，每层隐藏状态作为下一层输入。通过 `nn.RNN(num_layers=2)` 实现。

In [17]:
deep_rnn = nn.RNN(input_size=vocab_size, hidden_size=num_hiddens, num_layers=2)
print(f'num_layers = {deep_rnn.num_layers}, 参数个数: {sum(p.numel() for p in deep_rnn.parameters())}')
net_deep = RNNModel(deep_rnn, vocab_size)
train(net_deep, data_iter, 1, 500, device, vocab)

num_layers = 2, 参数个数: 204800
epoch 100, perplexity 2.8
epoch 200, perplexity 1.7
epoch 300, perplexity 1.6
epoch 400, perplexity 1.6
epoch 500, perplexity 1.6
predict: time traveller smiled round at us then still smiling faintly andw


## 4. 双向循环神经网络

同时利用序列的前后向信息。通过 `nn.RNN(bidirectional=True)` 实现，输出维度翻倍。

In [18]:
bi_rnn = nn.RNN(input_size=vocab_size, hidden_size=num_hiddens, bidirectional=True)
print(f'bidirectional = {bi_rnn.bidirectional}, 输出维度 = {num_hiddens * 2}')
net_bi = RNNModel(bi_rnn, vocab_size)
train(net_bi, data_iter, 1, 500, device, vocab)

bidirectional = True, 输出维度 = 512
epoch 100, perplexity 1.2
epoch 200, perplexity 1.2
epoch 300, perplexity 1.1
epoch 400, perplexity 1.2
epoch 500, perplexity 1.1
predict: time traveller r rererererererererererererererererererererererere


## 5. 机器翻译与数据集
利用循环神经网络，实现一个有Tatoeba项目的双语句子对组成的“英文-法文”翻译模型。